# Занятие 5, демо 2. Условные таргеты противоречат друг другу

Регрессия обучается на **условный** таргет $v^{\mathrm{VP}}(x_0,\epsilon,\tau)$:
он зависит от скрытой пары, а не только от того, что сеть видит на входе. Через
одну и ту же точку в один и тот же момент проходят разные пары - и таргеты у них
разные.

Что тогда выучивает сеть?

In [ ]:
import torch

torch.set_num_threads(1)

"""Конфликт условных таргетов и одно marginal-поле.

Одномерный пример, в котором всё считается руками: данные - две точки,
расписание то же угловое, что на занятии 5.
"""
import math

import torch
from torch import nn


def schedule(tau):
    phi = (math.pi / 2) * tau
    return torch.cos(phi), torch.sin(phi)


def vp_target(x_0, eps, tau):
    alpha, sigma = schedule(tau)
    return alpha * eps - sigma * x_0


def sample_two_point(n, generator):
    """Данные: плюс один и минус один с равной вероятностью."""
    signs = torch.randint(2, (n, 1), generator=generator, dtype=torch.float64)
    return 2.0 * signs - 1.0


class TinyNet(nn.Module):
    """Маленькая сеть: (x, tau) -> предсказание v^VP."""

    def __init__(self, width=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, width), nn.SiLU(),
            nn.Linear(width, width), nn.SiLU(),
            nn.Linear(width, 1),
        )

    def forward(self, x, tau):
        return self.net(torch.cat([x, tau], dim=-1))


def preimages(x_tau, tau):
    """Пары (x_0, eps), дающие эту точку в этот момент, и их таргеты.

    Требует 0 < tau <= 1: при tau=0 имеем sigma=0, шум по точке не
    восстанавливается и деление обращается в ноль.

    Для x_0 из двух точек шум восстанавливается однозначно: eps = (x - a x_0)/s.
    Возвращает список (x_0, eps, вес, таргет), где вес - апостериорная
    вероятность ветки: она пропорциональна плотности нужного шума, а общий
    множитель 1/sigma и равные априорные 1/2 сокращаются при нормировке.
    """
    if not 0 < tau <= 1:
        raise ValueError("preimages определён при 0 < tau <= 1")
    alpha, sigma = schedule(torch.tensor(tau, dtype=torch.float64))
    out = []
    for x_0 in (1.0, -1.0):
        eps = (x_tau - float(alpha) * x_0) / float(sigma)
        weight = math.exp(-0.5 * eps ** 2)
        target = float(alpha) * eps - float(sigma) * x_0
        out.append((x_0, eps, weight, target))
    total = sum(w for _, _, w, _ in out)
    return [(x_0, eps, w / total, t) for x_0, eps, w, t in out]


def marginal_target(x_tau, tau):
    """Population-оптимум квадратичной регрессии: E[v^VP | x_tau, tau]."""
    return sum(w * t for _, _, w, t in preimages(x_tau, tau))


def marginal_exact_half(x_tau):
    """То же самое при tau=1/2, выписанное вручную.

    При alpha=sigma=1/sqrt(2) отношение апостериорных вероятностей равно
    exp(2*sqrt(2)*x), поэтому E[x_0 | x] = tanh(sqrt(2)*x) и
    E[v^VP | x] = x - sqrt(2)*tanh(sqrt(2)*x). Нужно как независимая проверка
    численной ветки: обе считаются разными способами.
    """
    return x_tau - math.sqrt(2) * math.tanh(math.sqrt(2) * x_tau)


def fit_two_examples(steps=200, lr=2e-3, seed=0, x_tau=0.0, tau=0.5):
    """Регрессия ровно на двух примерах с одинаковым входом.

    Тот самый случай из тезиса: один и тот же (x_tau, tau) с двумя
    противоположными таргетами. В обычном обучении два буквально одинаковых
    входа не встречаются - здесь мы их подаём руками.

    Возвращает список (шаг, выход сети, потери).
    """
    torch.manual_seed(seed)
    model = TinyNet().double()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    rows = [(x_0, t) for x_0, _, _, t in preimages(x_tau, tau)]
    x = torch.full((len(rows), 1), x_tau, dtype=torch.float64)
    moment = torch.full((len(rows), 1), tau, dtype=torch.float64)
    y = torch.tensor([[t] for _, t in rows], dtype=torch.float64)

    history = []
    for step in range(steps + 1):
        prediction = model(x, moment)
        loss = ((prediction - y) ** 2).mean()
        history.append((step, float(prediction[0]), float(loss)))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    return history


def train_tiny(steps=6000, batch=1024, lr=2e-3, seed=0):
    """Обучение с квадратичной ошибкой на полном распределении."""
    torch.manual_seed(seed)
    model = TinyNet().double()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed + 1)
    history = []

    for _ in range(steps):
        x_0 = sample_two_point(batch, g)
        eps = torch.randn(batch, 1, generator=g, dtype=torch.float64)
        tau = torch.rand(batch, 1, generator=g, dtype=torch.float64)
        alpha, sigma = schedule(tau)
        prediction = model(alpha * x_0 + sigma * eps, tau)
        loss = ((prediction - vp_target(x_0, eps, tau)) ** 2).mean()
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        history.append(loss.item())

    return model, history

## Самый ясный случай

Данные - две точки: $x_0=+1$ и $x_0=-1$ с равной вероятностью. Момент
$\tau=\tfrac12$, значит $\alpha=\sigma=\tfrac1{\sqrt2}$.

Какие пары дают точку $x_\tau=0$?

In [ ]:
print(f"{'x_0':>6}  {'eps':>9}  {'вес':>8}  {'таргет v^VP':>13}")
for x_0, eps, weight, target in preimages(0.0, 0.5):
    print(f"{x_0:>+6.1f}  {eps:>+9.4f}  {weight:>8.4f}  {target:>+13.4f}")

## Подадим эти два примера буквально

В обычном обучении два совпадающих входа не встречаются: $\epsilon$ и $\tau$
непрерывны. Поэтому подадим их руками - один и тот же вход $(0,\tfrac12)$ с двумя
противоположными таргетами - и посмотрим, что сделает квадратичная ошибка.

In [ ]:
print(f"{'шаг':>5}  {'выход сети':>13}  {'потери':>11}")
for step, output, loss in fit_two_examples(steps=200):
    if step in (0, 50, 100, 200):
        print(f"{step:>5}  {output:>+13.6f}  {loss:>11.6f}")

## Что здесь произошло

Сеть не выбрала ни один из двух таргетов - она пришла к их среднему, то есть к
нулю. И потери встали на $2$: это в точности
$\tfrac12\bigl((\sqrt2)^2+(-\sqrt2)^2\bigr)$, разброс самих таргетов вокруг их
среднего. Меньше здесь не сделает никакая сеть - одному входу нельзя сопоставить
два разных числа.

Теперь общий случай. Прообразов у точки может быть много, и они не равноправны:
чем менее вероятен нужный шум, тем меньше вес ветки. Population-оптимум
квадратичной регрессии - **условное среднее** с этими весами:

$$
\bar v(x_\tau,\tau)=\mathbb E\bigl[v^{\mathrm{VP}}\mid x_\tau,\tau\bigr]
$$

В точке $x_\tau=0$ веса были равны, и среднее совпало с полусуммой. В
несимметричной точке - уже нет.

In [ ]:
for x_0, eps, weight, target in preimages(0.3, 0.5):
    print(f"  x_0={x_0:>+4.1f}  вес {weight:.4f}  таргет {target:>+8.4f}")

plain = sum(t for *_, t in preimages(0.3, 0.5)) / 2
print(f"\n  полусумма таргетов:  {plain:>+8.4f}")
print(f"  условное среднее:    {marginal_target(0.3, 0.5):>+8.4f}")
print(f"  оно же вручную:      {marginal_exact_half(0.3):>+8.4f}"
      f"   [x - sqrt(2)*tanh(sqrt(2)x)]")

## Полное обучение

Теперь обычное обучение на полном распределении - и сравнение с точным условным
средним. Рядом печатаем, насколько потери сети выше неустранимых: само по себе
значение лосса ни о чём не говорит, значение имеет только этот зазор.

In [ ]:
model, _ = train_tiny()

g = torch.Generator().manual_seed(4242)
x_0 = sample_two_point(20_000, g)
eps = torch.randn(20_000, 1, generator=g, dtype=torch.float64)
tau = torch.rand(20_000, 1, generator=g, dtype=torch.float64)
alpha, sigma = schedule(tau)
x_tau = alpha * x_0 + sigma * eps
v = vp_target(x_0, eps, tau)

exact = torch.tensor([[marginal_target(float(a), float(b))]
                      for a, b in zip(x_tau, tau)], dtype=torch.float64)
with torch.no_grad():
    net_loss = float(((model(x_tau, tau) - v) ** 2).mean())
best_loss = float(((exact - v) ** 2).mean())

print(f"потери сети {net_loss:.4f}, неустранимые {best_loss:.4f}, "
      f"лишнее {net_loss - best_loss:.4f}")
print()
print(f"{'x':>6}  {'сеть':>10}  {'условное среднее':>18}  {'разница':>10}")
with torch.no_grad():
    for x in (-0.7, -0.3, 0.0, 0.3, 0.7):
        point = torch.tensor([[x]], dtype=torch.float64)
        moment = torch.tensor([[0.5]], dtype=torch.float64)
        got = float(model(point, moment))
        want = marginal_target(x, 0.5)
        print(f"{x:>+6.1f}  {got:>+10.4f}  {want:>+18.4f}  {abs(got - want):>10.4f}")

## Что из этого следует

Первый опыт показал механизм в чистом виде: одинаковый вход, противоположные
таргеты - квадратичная ошибка выбирает среднее, и потери остаются положительными.
Второй показал, что обученная на полном распределении сеть оказывается **близка**
к тому же условному среднему: зазор по потерям около 0.012 при неустранимых 0.64,
по точкам расхождение до пяти сотых. Именно «близка», а не «сходится»: шаг
обучения постоянный, и сеть продолжает колебаться вокруг оптимума.

Отсюда работает вся конструкция занятий 4-5. Обучать marginal-поле напрямую
нельзя - его значение в точке неизвестно. Но регрессия на доступный условный
таргет имеет тот же population-оптимум, потому что этот оптимум и есть условное
среднее условного таргета.

И про потери. Ненулевой лосс здесь не признак плохого обучения: неустранимая
часть - это $\operatorname{Var}(v^{\mathrm{VP}}\mid x_\tau,\tau)$, и она
положительна. Судить по абсолютному значению лосса нельзя, судить можно по зазору
до неустранимой части - как в строке выше.